<a href="https://www.kaggle.com/code/kedhareswernaidu/moonknight?scriptVersionId=231387099" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Like Paintings

### Environment Setup and Imports

In [15]:
pip install tensorflow-addons==<compatible_version>

/bin/bash: -c: line 1: syntax error near unexpected token `newline'
/bin/bash: -c: line 1: `/usr/bin/python3 -m pip install tensorflow-addons==<compatible_version>'
Note: you may need to restart the kernel to use updated packages.


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, Model, initializers

# --------------------------
# 1. Self-Attention Layer (Fixed)
# --------------------------
class SelfAttention(layers.Layer):
    def __init__(self, filters):
        """ Self-Attention Layer for capturing global dependencies. """
        super(SelfAttention, self).__init__()
        self.filters = filters
        self.query = layers.Conv2D(filters // 8, (1, 1), padding="same")
        self.key = layers.Conv2D(filters // 8, (1, 1), padding="same")
        self.value = layers.Conv2D(filters, (1, 1), padding="same")
        self.gamma = self.add_weight(shape=(1,), initializer="zeros", trainable=True)

    def call(self, x):
        batch_size, height, width, channels = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], tf.shape(x)[3]

        q = tf.reshape(self.query(x), (batch_size, height * width, channels // 8))  # [B, HW, C/8]
        k = tf.reshape(self.key(x), (batch_size, height * width, channels // 8))    # [B, HW, C/8]
        v = tf.reshape(self.value(x), (batch_size, height * width, channels))       # [B, HW, C]

        attention_map = tf.nn.softmax(tf.matmul(q, k, transpose_b=True))  # [B, HW, HW]
        out = tf.matmul(attention_map, v)  # [B, HW, C]

        out = tf.reshape(out, (batch_size, height, width, channels))  # Restore original shape
        return self.gamma * out + x  # Weighted residual connection

# --------------------------
# 2. Adaptive Instance Normalization (AdaIN)
# --------------------------
class AdaIN(layers.Layer):
    def __init__(self, epsilon=1e-5):
        super(AdaIN, self).__init__()
        self.epsilon = epsilon

    def call(self, x):
        mean, var = tf.nn.moments(x, axes=[1, 2], keepdims=True)
        std = tf.sqrt(var + self.epsilon)
        return (x - mean) / std

# --------------------------
# 3. ResNet Block with AdaIN and Self-Attention
# --------------------------
def resnet_block(filters):
    def block(x):
        res = layers.Conv2D(filters, (3, 3), padding="same")(x)
        res = AdaIN()(res)
        res = layers.ReLU()(res)
        res = SelfAttention(filters)(res)  # Add Self-Attention
        res = layers.Conv2D(filters, (3, 3), padding="same")(res)
        res = AdaIN()(res)
        return layers.Add()([x, res])  # Residual Connection
    return block

# --------------------------
# 4. Generator Model (U-Net + ResNet)
# --------------------------
def build_generator():
    inputs = layers.Input(shape=(256, 256, 3))

    # Encoder
    x = layers.Conv2D(64, (7, 7), padding="same", strides=1)(inputs)
    x = AdaIN()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(128, (3, 3), padding="same", strides=2)(x)
    x = AdaIN()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(256, (3, 3), padding="same", strides=2)(x)
    x = AdaIN()(x)
    x = layers.ReLU()(x)

    # Residual Blocks
    for _ in range(6):
        x = resnet_block(256)(x)

    # Decoder
    x = layers.Conv2DTranspose(128, (3, 3), padding="same", strides=2)(x)
    x = AdaIN()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(64, (3, 3), padding="same", strides=2)(x)
    x = AdaIN()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(3, (7, 7), padding="same", activation="tanh")(x)

    return Model(inputs, x, name="Generator")

# --------------------------
# 5. Discriminator Model (PatchGAN)
# --------------------------
def build_discriminator():
    inputs = layers.Input(shape=(256, 256, 3))

    x = layers.Conv2D(64, (4, 4), strides=2, padding="same")(inputs)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv2D(128, (4, 4), strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv2D(256, (4, 4), strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv2D(512, (4, 4), strides=1, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv2D(1, (4, 4), strides=1, padding="same", activation="sigmoid")(x)

    return Model(inputs, x, name="Discriminator")

# --------------------------
# 6. Instantiate Models
# --------------------------
generator_G = build_generator()  # Photo → Monet
generator_F = build_generator()  # Monet → Photo
discriminator_X = build_discriminator()  # Discriminator for real vs fake photos
discriminator_Y = build_discriminator()  # Discriminator for real vs fake Monet paintings

print("✅ Models initialized successfully!")

✅ Models initialized successfully!


/usr/local/lib/python3.10/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [6]:
import tensorflow as tf
from tensorflow.keras import optimizers

# --------------------------
# 1. Adversarial Loss (Least Squares GAN)
# --------------------------
mse_loss = tf.keras.losses.MeanSquaredError()

def generator_loss(fake_output):
    """ Adversarial loss for generators (LS-GAN). """
    return mse_loss(tf.ones_like(fake_output), fake_output)  # Try to fool discriminator

def discriminator_loss(real_output, fake_output):
    """ Discriminator loss (LS-GAN). """
    real_loss = mse_loss(tf.ones_like(real_output), real_output)  # Real images should be 1
    fake_loss = mse_loss(tf.zeros_like(fake_output), fake_output)  # Fake images should be 0
    return (real_loss + fake_loss) * 0.5  # Average loss

# --------------------------
# 2. Cycle Consistency Loss (Content Preservation)
# --------------------------
mae_loss = tf.keras.losses.MeanAbsoluteError()

def cycle_loss(real_image, cycled_image, lambda_cycle=10):
    """ Ensures translated image can be reconstructed back to original. """
    return lambda_cycle * mae_loss(real_image, cycled_image)

# --------------------------
# 3. Identity Loss (Style Preservation)
# --------------------------
def identity_loss(real_image, same_image, lambda_identity=5):
    """ Encourages generators to retain color composition. """
    return lambda_identity * mae_loss(real_image, same_image)

# --------------------------
# 4. Optimizers (Separate for each model)
# --------------------------
generator_G_optimizer = optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
generator_F_optimizer = optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
discriminator_X_optimizer = optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
discriminator_Y_optimizer = optimizers.Adam(learning_rate=2e-4, beta_1=0.5)

print("✅ Loss functions & optimizers initialized!")

✅ Loss functions & optimizers initialized!


In [7]:
import tensorflow as tf

# --------------------------
# 1. Training Step Function
# --------------------------
@tf.function
def train_step(real_x, real_y):
    """Performs one training step: forward pass, compute losses, and backpropagation."""

    with tf.GradientTape(persistent=True) as tape:
        # -------------------------------
        # Forward Pass - Generators
        # -------------------------------
        fake_y = generator_G(real_x, training=True)  # X → Y
        cycled_x = generator_F(fake_y, training=True)  # Y' → X
        
        fake_x = generator_F(real_y, training=True)  # Y → X
        cycled_y = generator_G(fake_x, training=True)  # X' → Y

        # Identity Mapping (Preserve colors for images already in target domain)
        same_y = generator_G(real_y, training=True)  # Y → Y
        same_x = generator_F(real_x, training=True)  # X → X
        
        # -------------------------------
        # Forward Pass - Discriminators
        # -------------------------------
        disc_real_x = discriminator_X(real_x, training=True)  # Real X
        disc_fake_x = discriminator_X(fake_x, training=True)  # Fake X (generated)

        disc_real_y = discriminator_Y(real_y, training=True)  # Real Y
        disc_fake_y = discriminator_Y(fake_y, training=True)  # Fake Y (generated)

        # -------------------------------
        # Compute Losses
        # -------------------------------
        # Generator Adversarial Loss
        gen_G_loss = generator_loss(disc_fake_y)
        gen_F_loss = generator_loss(disc_fake_x)

        # Cycle Consistency Loss
        cycle_X_loss = cycle_loss(real_x, cycled_x)
        cycle_Y_loss = cycle_loss(real_y, cycled_y)
        total_cycle_loss = cycle_X_loss + cycle_Y_loss

        # Identity Loss
        id_X_loss = identity_loss(real_x, same_x)
        id_Y_loss = identity_loss(real_y, same_y)
        total_identity_loss = id_X_loss + id_Y_loss

        # Total Generator Loss
        total_gen_G_loss = gen_G_loss + total_cycle_loss + total_identity_loss
        total_gen_F_loss = gen_F_loss + total_cycle_loss + total_identity_loss

        # Discriminator Loss
        disc_X_loss = discriminator_loss(disc_real_x, disc_fake_x)
        disc_Y_loss = discriminator_loss(disc_real_y, disc_fake_y)

    # --------------------------
    # Compute Gradients
    # --------------------------
    generator_G_gradients = tape.gradient(total_gen_G_loss, generator_G.trainable_variables)
    generator_F_gradients = tape.gradient(total_gen_F_loss, generator_F.trainable_variables)
    discriminator_X_gradients = tape.gradient(disc_X_loss, discriminator_X.trainable_variables)
    discriminator_Y_gradients = tape.gradient(disc_Y_loss, discriminator_Y.trainable_variables)

    # --------------------------
    # Apply Gradients (Backpropagation)
    # --------------------------
    generator_G_optimizer.apply_gradients(zip(generator_G_gradients, generator_G.trainable_variables))
    generator_F_optimizer.apply_gradients(zip(generator_F_gradients, generator_F.trainable_variables))
    discriminator_X_optimizer.apply_gradients(zip(discriminator_X_gradients, discriminator_X.trainable_variables))
    discriminator_Y_optimizer.apply_gradients(zip(discriminator_Y_gradients, discriminator_Y.trainable_variables))

    return {
        "G_G_loss": total_gen_G_loss,
        "G_F_loss": total_gen_F_loss,
        "D_X_loss": disc_X_loss,
        "D_Y_loss": disc_Y_loss
    }

print("✅ Training step function initialized!")

✅ Training step function initialized!


In [13]:
import tensorflow as tf
import tensorflow_addons as tfa
import glob
import random
import os

# --------------------------
# 1. Dataset Paths
# --------------------------
PHOTO_DIR = "/kaggle/input/c/gan-getting-started/photo_jpg"  
MONET_DIR = "/kaggle/input/c/gan-getting-started/monet_jpg" 

# --------------------------
# 2. Image Preprocessing Function
# --------------------------
def preprocess_image(image):
    """Resizes image to 256x256 and normalizes pixel values to [-1, 1]."""
    image = tf.image.resize(image, (256, 256))
    image = (image - 127.5) / 127.5  # Normalize to [-1, 1]
    return image

# --------------------------
# 3. Data Augmentation
# --------------------------
def augment_image(image):
    """Applies random jitter and flip to the image."""
    image = tf.image.random_flip_left_right(image)  # Horizontal flip
    image = tf.image.random_saturation(image, 0.8, 1.2)  # Random saturation
    image = tf.image.random_brightness(image, 0.1)  # Random brightness
    return image

# --------------------------
# 4. Load Dataset from Files
# --------------------------
def load_image(filename):
    """Loads and preprocesses image from file."""
    image = tf.io.read_file(filename)
    image = tf.image.decode_jpeg(image, channels=3)
    image = preprocess_image(image)
    return image

# --------------------------
# 5. Build TensorFlow Dataset Pipeline
# --------------------------
def get_dataset(image_paths, batch_size=1, augment=False):
    """Creates a TensorFlow dataset pipeline with optional augmentation."""
    dataset = tf.data.Dataset.from_tensor_slices(image_paths)
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.shuffle(len(image_paths)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

# --------------------------
# 6. Load Datasets
# --------------------------
photo_paths = glob.glob(os.path.join(PHOTO_DIR, "*.jpg"))
monet_paths = glob.glob(os.path.join(MONET_DIR, "*.jpg"))

photo_ds = get_dataset(photo_paths, batch_size=1, augment=True)
monet_ds = get_dataset(monet_paths, batch_size=1, augment=True)

print(f"✅ Loaded {len(photo_paths)} photo images and {len(monet_paths)} Monet images.")

ModuleNotFoundError: No module named 'keras.src.engine'